### Importando bibliotecas

In [2]:
import pandas as pd

### Lendo os dados

In [2]:
df = pd.read_csv("../data/BMR_Dataset.csv")
df.head()

,user_id,age,weight,height,gender,BMR
0,1243,27.0,73.066833,162.723887,Male,1576.878448
1,4971,28.0,77.730284,179.495414,Female,1536.656455
2,1629,57.0,85.704790,158.403052,Male,1697.694955
3,6201,47.0,67.012186,168.113746,Male,1471.524942
4,7833,36.0,79.929512,175.561126,Male,1743.147435


### Removendo colunas desnecessarias

In [3]:
df.drop(columns=["user_id"], inplace=True)
df.head()

,age,weight,height,gender,BMR
0,27.0,73.066833,162.723887,Male,1576.878448
1,28.0,77.730284,179.495414,Female,1536.656455
2,57.0,85.704790,158.403052,Male,1697.694955
3,47.0,67.012186,168.113746,Male,1471.524942
4,36.0,79.929512,175.561126,Male,1743.147435


### Separando features de labels

In [4]:
X, y = df.drop(columns=["BMR"]), df["BMR"]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.head()

,age,weight,height,gender
6317,32.0,72.380435,182.735573,Male
740,47.0,73.664635,159.290008,Male
3781,27.0,72.861273,176.984767,Female
7850,51.0,102.980871,147.693695,Male
2963,27.0,55.730900,170.618041,Male


In [6]:
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

### Tratamento de dados categoricos

In [8]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    sparse_output=False,
)
X_cat = ohe.fit_transform(
    X_train.select_dtypes(include=["object"])
)
X_cat_df = pd.DataFrame(
    X_cat, 
    columns=ohe.get_feature_names_out(),
    index=X_train.index,
)
X_cat_df.head()

,gender_Female,gender_Male,gender_nan
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,1.0,0.0
4,0.0,1.0,0.0


In [10]:
X_train = pd.concat([
    X_cat_df, 
    X_train.select_dtypes(include=["number"])
], axis=1)

In [11]:
X_train.head()

,gender_Female,gender_Male,gender_nan,age,weight,height
0,0.0,1.0,0.0,32.0,72.380435,182.735573
1,0.0,1.0,0.0,47.0,73.664635,159.290008
2,1.0,0.0,0.0,27.0,72.861273,176.984767
3,0.0,1.0,0.0,51.0,102.980871,147.693695
4,0.0,1.0,0.0,27.0,55.730900,170.618041


### Normalização dos dados

In [12]:
from sklearn.preprocessing import (
    StandardScaler
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_train.columns)
X_scaled_df.head()

,gender_Female,gender_Male,gender_nan,age,weight,height
0,-0.998057,1.0,-0.031196,-0.665347,0.183038,1.307387
1,-0.998057,1.0,-0.031196,0.439434,0.268493,-1.068944
2,1.001946,-1.0,-0.031196,-1.033607,0.215034,0.724513
3,-0.998057,1.0,-0.031196,0.734042,2.219309,-2.244291
4,-0.998057,1.0,-0.031196,-1.033607,-0.924887,0.079211


In [13]:
X_scaled_df.shape

(7200, 6)

#### Treinando o modelo

In [15]:
import lightgbm as lgb

lgb_regressor = lgb.LGBMRegressor()
lgb_regressor.fit(X_scaled, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000281 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 565
[LightGBM] [Info] Number of data points in the train set: 7200, number of used features: 5
[LightGBM] [Info] Start training from score 1474.868216


LGBMRegressor()

### Validacao

In [16]:
X_test_cat = ohe.transform(
    X_test.select_dtypes(include=["object"])
)
X_test_cat_df = pd.DataFrame(
    X_test_cat, 
    columns=ohe.get_feature_names_out(),
    index=X_test.index,
)
X_test_cat_df.head()

,gender_Female,gender_Male,gender_nan
0,0.0,1.0,0.0
1,1.0,0.0,0.0
2,1.0,0.0,0.0
3,0.0,1.0,0.0
4,0.0,1.0,0.0


In [17]:
X_test = pd.concat([
    X_test_cat_df, 
    X_test.select_dtypes(include=["number"])
], axis=1)

In [18]:
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(
    X_test_scaled, 
    columns=X_test.columns
)
X_test_scaled_df.head()

,gender_Female,gender_Male,gender_nan,age,weight,height
0,-0.998057,1.0,-0.031196,-0.959955,-0.346936,-0.701732
1,1.001946,-1.0,-0.031196,-0.002479,0.290256,0.349294
2,1.001946,-1.0,-0.031196,-0.002479,-0.086603,-0.408357
3,-0.998057,1.0,-0.031196,0.071173,-0.942905,0.608041
4,-0.998057,1.0,-0.031196,-0.591695,-1.447337,-1.755304


In [20]:
y_hat = lgb_regressor.predict(X_test_scaled_df)

In [23]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_hat)
mape = mean_absolute_percentage_error(y_test, y_hat)
mse = mean_squared_error(y_test, y_hat)
r2 = r2_score(y_test, y_hat)

print(f"MAE: {mae}")
print(f"MAPE: {100 * mape} %")
print(f"MSE: {mse}")
print(f"R2: {r2}")

MAE: 42.46175584886078
MAPE: 2.9424334880404834 %
MSE: 2848.990313362008
R2: 0.9282841368954043


### FLAML

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/BMR_Dataset.csv")
df.head()

,user_id,age,weight,height,gender,BMR
0,1243,27.0,73.066833,162.723887,Male,1576.878448
1,4971,28.0,77.730284,179.495414,Female,1536.656455
2,1629,57.0,85.704790,158.403052,Male,1697.694955
3,6201,47.0,67.012186,168.113746,Male,1471.524942
4,7833,36.0,79.929512,175.561126,Male,1743.147435


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["BMR"]), 
    df["BMR"], 
    test_size=0.2, 
    random_state=42
)


In [5]:
X_train.head()

,user_id,age,weight,height,gender
6317,4226,32.0,72.380435,182.735573,Male
740,6767,47.0,73.664635,159.290008,Male
3781,7418,27.0,72.861273,176.984767,Female
7850,8474,51.0,102.980871,147.693695,Male
2963,7706,27.0,55.730900,170.618041,Male


In [7]:
import flaml

flaml_regressor = flaml.AutoML()
automl_settings = {
    "time_budget": 1,  # in seconds
    "metric": "r2",
    "task": "regression",
    "log_file_name": "california.log",
}
flaml_regressor.fit(X_train, y_train, **automl_settings)
y_hat = flaml_regressor.predict(X_test)

[flaml.automl.logger: 04-23 22:21:33] {1728} INFO - task = regression
[flaml.automl.logger: 04-23 22:21:33] {1739} INFO - Evaluation method: holdout
[flaml.automl.logger: 04-23 22:21:33] {1838} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 04-23 22:21:33] {1955} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd']
[flaml.automl.logger: 04-23 22:21:33] {2258} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 04-23 22:21:33] {2393} INFO - Estimated sufficient time budget=189s. Estimated necessary time budget=1s.
[flaml.automl.logger: 04-23 22:21:33] {2442} INFO -  at 0.1s,	estimator lgbm's best error=0.6854,	best estimator lgbm's best error=0.6854
[flaml.automl.logger: 04-23 22:21:33] {2258} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 04-23 22:21:33] {2442} INFO -  at 0.1s,	estimator lgbm's best error=0.6854,	best estimator lgbm's best error=0.6854
[flaml.automl.logger: 04-23 22:21:33] {225

In [8]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_hat)
mape = mean_absolute_percentage_error(y_test, y_hat)
mse = mean_squared_error(y_test, y_hat)
r2 = r2_score(y_test, y_hat)

print(f"MAE: {mae}")
print(f"MAPE: {100 * mape} %")
print(f"MSE: {mse}")
print(f"R2: {r2}")

MAE: 42.17137046010104
MAPE: 2.9186869819185417 %
MSE: 2831.985284774734
R2: 0.9287121939149509


In [9]:
flaml_regressor.predict(
    pd.DataFrame(
        {
            "user_id": [1],
            "age": [27],
            "weight": [70],
            "height": [185],
            "gender": ["Male"],
        }
    )
)

array([1745.86720761])